In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import catboost as cb
import lightgbm as lgb
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

from mllabs import Experimenter, Connector, ColSelector, ProgressSessionLogger, TqdmProgressSession
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first
from mllabs.collector import SHAPCollector
from mllabs.filter import RandomFilter
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor, CrossFitTransformer
from mllabs.collector import MetricCollector, ModelAttrCollector

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score

2026-05-12 00:23:29.110985: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 00:23:29.148185: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-12 00:23:30.208420: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

In [2]:
data_path = Path('data')

dict_expr = {
    'Mulching_Used_n': (pl.col('Mulching_Used') == 'Yes').cast(pl.Int8),
    'Soil_25': (pl.col('Soil_Moisture') < 25).cast(pl.Int8),
    'Temp_30': (pl.col('Temperature_C') > 30).cast(pl.Int8),
    'Rain_300': (pl.col('Rainfall_mm') < 300).cast(pl.Int8),
    'Wind_10': (pl.col('Wind_Speed_kmh') > 10).cast(pl.Int8),
}

loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr),
)

df_train = loader.fit_transform([data_path / 'train.csv'])
df_test = loader.transform([data_path / 'test.csv'])

y = 'Irrigation_Need'
X_cat = ['Crop_Growth_Stage', 'Crop_Type', 'Irrigation_Type', 'Region', 'Season', 'Soil_Type', 'Water_Source']
X_num = ['Electrical_Conductivity', 'Field_Area_hectare', 'Humidity', 'Organic_Carbon', 'Previous_Irrigation_mm', 'Rainfall_mm', 
         'Soil_Moisture', 'Soil_pH', 'Sunlight_Hours', 'Temperature_C','Wind_Speed_kmh']
X_bin = ['Mulching_Used_n']
X_num_bin = ['Soil_25', 'Temp_30', 'Rain_300', 'Wind_10']
X_all = X_num + X_cat + X_bin + X_num_bin

y_repl = {'Low': 0, 'Medium': 1, 'High': 2}
y_weight = {'Low': 1.000000, 'Medium': 1.547291, 'High': 17.607549}
df_train = df_train.with_columns(
    **{
        y: pl.col(y).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y).cast(pl.String).replace(y_weight).cast(pl.Float32)
    }
)

In [ ]:
if not os.path.exists('exp/modeling'):
    skf1 = StratifiedKFold(5, random_state = 123, shuffle=True)
    sss2 = StratifiedShuffleSplit(n_splits = 1, random_state = 123, train_size = 0.9)
    ex = Experimenter.create(
        df_train, 'exp/modeling', sp = sss1, sp_v = sss2, splitter_params={'y': y}, 
        logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))
    ex.set_grp('clf', method = 'predict', role = 'head', edges = {'y': [(None, y)], 'sample_weight': [(None, 'sample_weight')]})
    ex.set_grp(
        'xgb', parent = 'clf', processor=xgb.XGBClassifier, adapter=XGBoostAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'enable_categorical': True, 'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'})
    ex.add_collector(ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result'))
    ex.set_grp(
        'lgb', parent = 'clf', processor=lgb.LGBMClassifier, adapter=LightGBMAdapter(eval_mode='both'), 
        params={'random_state': 123, 'n_estimators': 10000, 'learning_rate': 0.05, 'verbose': -1, 'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},'eval_metric': 'multi_logloss'})
    ex.add_collector(
        ModelAttrCollector('lgb_evals_results', Connector(processor=lgb.LGBMClassifier), 'evals_result'))
    ex.set_grp(
        'cb', parent = 'clf', processor=cb.CatBoostClassifier, adapter=CatBoostAdapter(eval_mode='valid'),
        params={'early_stopping_rounds': 50, 'verbose': 0, 'random_state': 123, 'cat_features': ColSelector(col_type='category')})
    ex.add_collector(
        ModelAttrCollector('cb_evals_results', Connector('_base$', processor=cb.CatBoostClassifier), 'evals_result'))
    ex.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10})
    ex.add_collector(
        ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result'))
    ex.set_grp('lr', parent='clf', processor=LogisticRegression)

    ex.set_grp('pre', role = 'stage', method='transform')
    ex.set_grp('pre_ft', role = 'stage', method='fit_transform', edges = {'y': [(None, y)]})
    ex.set_node('std', grp='pre', processor=StandardScaler, edges={'X': [(None, X_num)]})
    ex.set_node('ohe', grp='pre', processor=OneHotEncoder, edges={'X': [(None, X_cat)]}, params={'sparse_output': False})

    # For catboost to be considered as Categorical Type
    e_fe.set_node('X_num_bin2str', grp='pre', edges={'X': [(None, X_num_bin)]}, processor=TypeConverter, params={'to':'str'})
    e_fe.set_node('X_num_binstr2cat', grp='pre', edges={'X': [('X_num_bin2str', None)]}, processor=CatConverter)
    ex.build()
    
    y_edges = {'y': [(None, y)]}
    ex.add_collector(
        MetricCollector('bAcc', Connector(edges = y_edges, role = 'head'), slice(-1, None), balanced_accuracy_score, include_train = True))
else:
    ex = Experimenter.load('exp/model', df_train, logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession))

In [ ]:
ex.set_node('xgb_X_lr_cf', grp='xgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
ex.set_node('lgb_X_lr_cf', grp='lgb', edges={'X': [(None, X_num + X_cat + X_bin + X_num_bin), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': None})
ex.set_node('cb_X_lr_cf', grp='cb', edges={'X': [(None, X_num + X_num_bin + X_cat + ['Mulching_Used']), ('X_lr_cf', None)]}, params={'n_estimators': 10000, 'gpu': 'yes'})
ex.set_node('nn_lr_cf', grp='nn', edges={'X': [('std', None), ('ohe', ohe_drop_first), (None, X_bin), (None, X_num_bin), ('X_lr_cf', slice(0, -1))]}, params={'epochs': 200, 'gpu': 'yes'})
ex.exp(finalize=True, n_jobs=2, gpu_id_list=[0])